In [2]:
// livesout.exe の基本情報を出力する
use std::process::Command;

let output = Command::new("powershell")
    .arg("-Command")
    .arg("Get-Item livesout.exe | Select-Object Name, Length, LastWriteTime")
    .output()
    .expect("Failed to get file info");

println!("{}", String::from_utf8_lossy(&output.stdout));


Name         Length LastWriteTime     
----         ------ -------------     
livesout.exe  10752 2026/05/05 6:04:10





## Case Study] ksnctf #27 Lives out - リバースエンジニアリング記録
### 1. 目的
- livesout.exe におけるクリア判定（描画ループ）をバイパスし、フラグを表示する TextOutW 命令まで実行フローを強制的に誘導する。

### 2. 実行環境
- Debugger: Visual Studio 2013 (x64 Debugger)
- Target: livesout.exe (x64)
- OS: Windows (ASLR有効)

### 3. 発生した問題と解決策
#### ① アドレス指定時の「無効な8進数」エラー
- 事象: アドレス（例: 00007ff6cf771559）を直接入力してジャンプしようとするとエラーが発生。

- 原因: 先頭の 0 により、Visual Studioが数値を8進数リテラルとして誤認したため。

- 対策: 16進数接頭辞 0x を付与し、0x00007ff6cf771559 と入力することで解決。

#### ② dbghook.c の出現と強制終了
- 事象: je 命令（判定ジャンプ）をスキップして「続行」した際、dbghook.c の検索ダイアログが出現し、その後アプリが終了した。

- 原因:パッチ（実行箇所の変更）により、プログラムが予期しないメモリ状態やレジストリ値で後続の処理（描画ループ等）を実行したため。 C Runtime (CRT) のデバッグチェックに抵触した可能性。

- 対策:呼び出し履歴 から livesout.exe のコードへ復帰する。一気に実行（F5）せず、F10（ステップオーバー） を使用して1行ずつ挙動を確認しながら進める。

### 4. 解析メモ：
重要なアドレスとオフセットASLRによりベースアドレスは変動するが、下位12ビット（末尾3〜4桁）のオフセットは不変

|命令の役割	|基準オフセット	|備考|
|:---|---|:---|
|描画ループ1 (jl)	|0x14D4	|NOP化の候補1|
|描画ループ2 (jl)	|0x14F3	|NOP化の候補2|
|クリア判定 (test)	|0x155E	|r11b の値をチェック|
|判定ジャンプ (je)	|0x1561	|最重要パッチ箇所 (0x771745へ飛ぶのを防ぐ)|
|文字出力 (TextOutW)	|0x173F	|ここを通過すればフラグが表示される|

### 5. 解決フェーズ：メッセージループの突破と動的パッチ
#### ① メッセージループからの脱出
- 事象: 0x11C0 付近の GetMessageW / DispatchMessageW ループで実行が止まり、先に進めなくなった。
- 原因: プログラムがユーザー入力を待機する「メッセージ待ち」状態であり、ステップ実行では抜け出せない。
- 対策:目的の描画処理（0x14D4 や 0x1561）にブレークポイントを設置。F5（実行）を押し、アプリのウィンドウを動かす（再描画イベントを発生させる）ことで、目的のアドレスで停止させた。

#### ② 描画処理の破壊 (Rectangleループのバイパス)
パズルを描画する処理を無効化し、フラグ生成処理へ強制遷移させるために以下のパッチを適用。

|アドレス (オフセット)|元の命令|修正内容 (NOP化)|目的|
|:---|---|:---|:---|
|0x14D4|jl |0x1480|90 90|横方向の描画ループを突破|
|0x14F3|jl |0x1470|90 90 90 90 90 90|縦方向の描画ループを突破||
|0x14FE|call SelectObject|90 90 90 90 90 90|不正レジスタによるクラッシュ防止
|0x1509|call SelectObject|90 90 90 90 90 90|同上

#### ③ クリア判定の書き換え
- アドレス: 0x1561 (ベースアドレス + 0x1561)
- 修正前: je 0x1745 (クリアしていない場合に終了処理へジャンプ)
- 修正後: 90 90 90 90 90 90 (NOP)
- 結果: 判定を無視して 0x1567 以降のフラグ生成・デコード処理が動き出した。

### 6. 最終結果
0x173F の call TextOutW が実行され、アプリのウィンドウ上に FLAG が表示された。

デバッガ（Visual Studio）の逆アセンブル画面が更新されない場合は、「次のステートメントに設定」 を行うことでメモリの変更が反映されることを確認。